## Colab setup

Auto-detects whether this is running in Colab. If so, clones the repo and reconstructs `.env` from a Colab secret. If running locally, this is a no-op (assumes the repo is already checked out and `.env` already exists).

In [1]:
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os
    repo_dir = "/content/change-my-view-2025"
    work_dir = repo_dir + "/this work"
    if not os.path.exists(repo_dir):  # skip re-cloning if this cell runs twice in the same session
        !git clone -b final-project https://github.com/jct-nlp/change-my-view-2025.git {repo_dir}
    if os.getcwd() != work_dir:  # skip re-cd-ing if already there
        %cd $work_dir
    with open("../.env", "w") as f:
        f.write(f"GEMINI_API_KEY={userdata.get('GEMINI_API_KEY')}")
else:
    print("Not running in Colab — skipping repo clone and .env setup (assuming local checkout already has both).")

Not running in Colab — skipping repo clone and .env setup (assuming local checkout already has both).


# 03.2 — Feature Engineering (Sections 6.5–6.9)

Continues from `03.1`'s checkpoint (`comments_df` with the 6.1–6.4 features, plus the running `untrusted_features` list) and computes the rest of section 6: 6.5–6.7 (CPU-only, moved here to keep `03.1` to a manageable size) and 6.8–6.9 (GPU-heavy — persuasive-language classification and BERT document embeddings with PCA reduction). Hands off to `03.3` for the OP-relation features and the final combine step.

> Loads `results/comments_df_for_03_2_checkpoint.csv` and `results/untrusted_features_for_03_2_checkpoint.json` (output of `03.1`). Needs a GPU runtime for 6.8/6.9 (Runtime → Change runtime type → GPU in Colab).

In [ ]:
import pandas as pd, json

comments_df = pd.read_csv('../results/comments_df_for_03_2_checkpoint.csv')
with open('../results/untrusted_features_for_03_2_checkpoint.json') as f:
    untrusted_features = json.load(f)

print(comments_df.shape)
print(untrusted_features)

In [ ]:
import common_functions

## 6.5 Adjective-Adverb Ratio

The **Adjective-Adverb Ratio**, also called the **Descriptive Density Ratio** measures the proportion of adjectives and adverbs in a text relative to the total word count. A higher ratio indicates a more expressive or emotionally charged writing style, while a lower ratio suggests a more concise, factual, and objective tone.

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

In [ ]:
import inspect
print(inspect.getsource(common_functions.get_adjective_adverb_ratio))

get_adjective_adverb_ratio = common_functions.get_adjective_adverb_ratio

In [133]:
comments_df['adj_adv_ratio'] = None

In [ ]:
import numpy as np
common_functions.do_calc_feature(comments_df, "adj_adv_ratio", get_adjective_adverb_ratio)

We want to plot the results. Rows with no adverbs get an infinite ratio, which isn't usable for plotting or stats, so we cap them at a sentinel value just above the observed finite max. A small margin keeps these rows ranked as the highest without an oversized value skewing the chart's scale or the mean/std.

In [135]:
finite_max = comments_df['adj_adv_ratio'].replace(np.inf, np.nan).max()
sentinel = finite_max + 2  # small margin above the real max keeps infinite-ratio rows the highest without distorting the plot

print("The finite max of adj_adv_ratio is:", finite_max)
print("The sentinel value for infinite ratios is:", sentinel)

The finite max of adj_adv_ratio is: 15.0
The sentinel value for infinite ratios is: 17.0


In [136]:
comments_df['adj_adv_ratio'] = comments_df['adj_adv_ratio'].replace(np.inf, sentinel).astype(float)

comments_df.to_csv('../results/comments_df_adj_adv_ratio_checkpoint.csv', index=False)

A value of `sentinel` (currently 17.0, printed above) is not a real measurement — it marks a comment that has at least one adjective but zero adverbs, where the true ratio is undefined (division by zero, i.e. mathematically infinite). We cap all such comments at the same fixed sentinel purely so the column stays plottable and comparable; it should be read as "infinite/undefined ratio", not as "17 times more adjectives than adverbs".

In [ ]:
common_functions.plot_feature(comments_df, 'adj_adv_ratio', 'Adjective-Adverb Ratio', 1)

In [63]:
import numpy as np

# "Infinite" ratio: at least one adjective, zero adverbs detected - capped at `sentinel` (see explanation above)
infinite_ratio_rows = comments_df[comments_df['adj_adv_ratio'] == sentinel]
infinite_example_row = infinite_ratio_rows.iloc[0]
print(f"Example for 'infinite' Adj/Adv ratio (capped at {sentinel}, {len(infinite_ratio_rows)} such comments in the dataset):")
print(infinite_example_row['final_comment'])
print("-" * 20)

# Highest genuinely finite ratio (excludes the capped 'infinite' rows)
finite_ratio_rows = comments_df[comments_df['adj_adv_ratio'] < sentinel]
max_finite_row = finite_ratio_rows.loc[finite_ratio_rows['adj_adv_ratio'].idxmax()]
print(f"Example for highest finite Adj/Adv ratio ({max_finite_row['adj_adv_ratio']}):")
print(max_finite_row['final_comment'])
print("-" * 20)

min_adj_adv_ratio_row = comments_df.loc[comments_df['adj_adv_ratio'].idxmin()]
print(f"Example for low Adj/Adv ratio ({min_adj_adv_ratio_row['adj_adv_ratio']}):")
print(min_adj_adv_ratio_row['final_comment'])
print("-" * 20)


Example for 'infinite' Adj/Adv ratio (capped at 17.0, 382 such comments in the dataset):
Sure it does.  And he doesn't have to sell it for it to become liquid capital.  He can go to any bank like JPM and borrow against it at a low %, buy whatever the fuck he wants and not pay capital gains etc.  The "stock isn't realized capital" argument is bullshit when you look at how billionaires leverage their net worth.
--------------------
Example for highest finite Adj/Adv ratio (15.0):
Depends where you look. on a macro/resource level you have a point.

But if you look at countries like Japan their issue is they have such an old population, and there isn't enough young people to do all the necessary jobs to look after the elderly.

A lot of western countries are facing similar impending crisis. Mainly around how populations are aging and there isn't enough money in pension pots to support them.

Whilst it may seem good the population as a whole is decreasing. The birth rate decreasing comes wi

Generally, we can say that:
* High Adjective-to-Adverb Ratio
  * More adjectives than adverbs
  * The text is descriptive and vivid, often used in storytelling and persuasive writing to create strong imagery and appeal to emotions.
  * May indicate a focus on ethos (credibility) and pathos (emotion) by painting a compelling picture.

* Low Adjective-to-Adverb Ratio
  * More adverbs than adjectives
  * The text is action-driven and dynamic, emphasizing how actions are performed, often found in arguments, speeches, or instructional writing.
  * May indicate a focus on logos (logical reasoning) and pathos (emotional appeal) by emphasizing how actions happen.

* Balanced Ratio
  * More or less the same amount of adjectives and adverbs
  * The text is neutral and precise, commonly seen in academic, formal, or news writing, aiming for a mix of credibility, logic, and emotional appeal.
  * May indicate a mix of ethos (credibility), logos (logic), and pathos (emotion) in persuasion.



We see that in our dataset, the texts tend to have high ratio of Adjective-to-Adverb.

In [138]:
overall_rate = comments_df['is_convincing'].mean()
infinite_rate = comments_df.loc[comments_df['adj_adv_ratio'] == sentinel, 'is_convincing'].mean()
rest_rate = comments_df.loc[comments_df['adj_adv_ratio'] < sentinel, 'is_convincing'].mean()

print(f"Overall convincing rate: {overall_rate:.1%}")
print(f"Convincing rate among 'infinite' ratio comments (zero adverbs): {infinite_rate:.1%}")
print(f"Convincing rate among comments with a measurable (finite) ratio: {rest_rate:.1%}")


Overall convincing rate: 16.1%
Convincing rate among 'infinite' ratio comments (zero adverbs): 6.0%
Convincing rate among comments with a measurable (finite) ratio: 17.2%


**Insight:** the bar chart's raw counts are dominated by the overall class imbalance (non-convincing comments outnumber convincing ones throughout), which can hide whether the ratio itself relates to persuasiveness. Looking at the convincing rate instead of raw counts, the "infinite" ratio group (zero adverbs, `~9.6%` of all comments) stands out: its convincing rate is roughly a third of the rate seen among comments with a measurable ratio (~6% vs. ~17%, against an overall average of ~16%). This suggests that comments written with no adverbs at all - often short, blunt, unhedged statements - tend to be less persuasive, while the moderate ratio bins (which hold almost all of the data) don't show a strong trend among themselves.


## 6.6 Readability

Readability is a crucial feature in the feature engineering of convincing and non-convincing texts because it directly impacts how easily the audience can understand and engage with the message. A highly readable text is often more persuasive, as it allows the reader to quickly grasp the key points and emotional appeals without feeling overwhelmed or lost in complex language.

One of the widely used metrics for estimating readability is the **Flesch-Kincaid Readability Tests**.

The **Flesch-Kincaid Readability Tests** provide a score based on sentence length and word complexity, with the result indicating the U.S. school grade level required to understand the text. A lower score suggests a more accessible text, which is often more persuasive.

We will calculate:
* Flesch-Kincaid Grade Level: This score gives an approximation of the U.S. school grade level needed to understand the text. A higher value means more complex language.
* Flesch Reading Ease: This score indicates how easy the text is to read. Higher values (above 60) indicate easier text, while lower values suggest more complex, harder-to-read text.

In [ ]:
!pip install textstat

### Flesh Kincaid Grade Level

In [139]:
comments_df['fk_grade_level'] = None

In [ ]:
import textstat
common_functions.do_calc_feature(comments_df, "fk_grade_level", lambda text: textstat.flesch_kincaid_grade(text), remove_urls=True)

In [141]:
comments_df['fk_grade_level'].head()

0      10.295
1    7.934588
2       -1.45
3    8.091111
4    8.806096
Name: fk_grade_level, dtype: object

The datatype is object, we will convert it to a number.

In [142]:
comments_df['fk_grade_level_converted'] = pd.to_numeric(comments_df['fk_grade_level'], errors='coerce')

In [143]:
comments_df['fk_grade_level_converted'].describe()

count    3971.000000
mean        9.121506
std         3.717330
min        -3.400000
25%         7.033082
50%         8.957923
75%        11.059985
max        44.190000
Name: fk_grade_level_converted, dtype: float64

We see that there's an unexpected result of -3.4 in the minimum, let's inspect the comments that resulted a value below 0:

In [144]:
comments_df[comments_df['fk_grade_level_converted'] < 0]

,original_post,thread_text,final_comment,is_convincing,final_comment_id,cleaned_final_comment,cleaned_thread_text,cleaned_original_post,sentiment_vader,sentiment_llm,tone_google,ethos_score,pathos_score,logos_score,ethos_llm,pathos_llm,logos_llm,adj_adv_ratio,fk_grade_level,fk_grade_level_converted
2,CMV: Mike Bloomberg's campaign is proof that t...,NaN,I saved your post it was so good. Hear hear to...,0,fjlcge6,saved your post it was so good Hear hear to that,NaN,CMV Mike Bloombergs campaign is proof that the...,0.7804,1.0,Passionate,0.000000,0.000000,0.083333,0.0,0.8,0.0,1.0,-1.45,-1.450000
330,CMV: if great britain not giving the 13 US col...,NaN,Good bot.,0,g712ly1,Good bot,NaN,CMV if great britain not giving the 13 US colo...,0.4404,0.5,Neutral,0.000000,0.000000,0.000000,0.0,0.1,0.0,17.0,-3.01,-3.010000
337,CMV: if great britain not giving the 13 US col...,NaN,It’s not,0,g6yqsz5,Its not,NaN,CMV if great britain not giving the 13 US colo...,0.0000,0.0,Neutral,0.000000,0.000000,0.000000,0.0,0.0,0.1,17.0,-3.01,-3.010000
402,cmv: Ban on sleeping in vehicles is just targe...,NaN,"RIP, man. Seemed like a great dude.",0,g7gsxzl,RIP man Seemed like great dude,NaN,cmv Ban on sleeping in vehicles is just target...,0.7650,0.6,Other,0.142857,0.142857,0.000000,0.0,0.8,0.0,17.0,-1.06,-1.060000
432,CMV: The white teen who said the n word on a S...,NaN,This is a great point tbh,0,gidgtax,This is great point tbh,NaN,CMV The white teen who said the word on Snap...,0.6249,0.7,Neutral,0.166667,0.166667,0.000000,0.0,0.1,0.1,17.0,-1.45,-1.450000
453,"CMV: America desperately needs a young, sane p...",NaN,Hot take,0,g0o23kd,Hot take,NaN,CMV America desperately needs young sane pres...,0.0000,0.0,Sarcastic,0.000000,0.000000,0.000000,0.0,0.0,0.0,17.0,-3.01,-3.010000
545,CMV: It should be illegal for huge media compa...,NaN,YES,0,gy2ga6o,YES,NaN,CMV It should be illegal for huge media compan...,0.4019,0.3,Passionate,0.000000,0.000000,0.000000,0.0,1.0,0.0,17.0,-3.4,-3.400000
620,CMV: People who have been wrongfully imprisone...,NaN,That's fucked,0,dz4clyw,Thats fucked,NaN,CMV People who have been wrongfully imprisoned...,-0.6597,-0.4,Passionate,0.000000,0.000000,0.000000,0.0,0.8,0.0,17.0,-3.01,-3.010000
732,CMV: taxpayers and the general public should n...,NaN,[Cardale Jones ain’t come here to play school....,0,dq5pp93,Cardale Jones aint come here to play schoolhtt...,NaN,CMV taxpayers and the general public should ne...,0.3400,0.0,Sarcastic,0.000000,0.000000,0.000000,0.2,0.6,0.3,1.0,-0.67,-0.670000
790,CMV: The modern remakes of older Disney movies...,NaN,You are wrong.,0,eafpem4,You are wrong,NaN,CMV The modern remakes of older Disney movies ...,-0.4767,-0.3,Other,0.000000,0.000000,0.000000,0.0,0.2,0.0,17.0,-2.62,-2.620000


We see that these are very short texts with very few syllables, so the result is expected. Let's inspect the distribution.

In [ ]:
common_functions.plot_feature(comments_df, 'fk_grade_level_converted', 'Flesh Kincaid Grade Level', 5)

**Insight:** the convincing rate isn't monotonic with grade level. Very simple comments (grade -3.4 to 1.6, ~90 comments) convince only ~2% of the time, and the low-complexity range up to grade ~6.6 sits at ~10% - both well under the ~16% overall average. The rate climbs through the bulk of the data (grade 6.6-11.6, ~60% of all comments) to ~17%, and peaks in the grade 11.6-16.6 range at ~23%, noticeably more persuasive than average. Past grade ~17 it falls again, and grades above ~20 have only a handful of comments each - too few to draw a conclusion. So there's a real sweet spot around high-school-to-early-college complexity, while very simple, short comments are the least persuasive group on this feature.

In [147]:
comments_df['fk_grade_level'] = comments_df['fk_grade_level_converted']
comments_df.drop(columns=['fk_grade_level_converted'], inplace=True)

In [157]:
example_targets = [0, 5, 9, 14, 18]
for target in example_targets:
    candidates = comments_df[
        comments_df['fk_grade_level'].between(target - 0.3, target + 0.3) &
        comments_df['final_comment'].str.len().between(60, 260)
    ]
    if len(candidates):
        row = candidates.iloc[0]
        print(f"Example near grade level {target} (actual: {row['fk_grade_level']:.2f}):")
        print(row['final_comment'])
        print("-" * 20)

max_grade_level_row = comments_df.loc[comments_df['fk_grade_level'].idxmax()]
print(f"Example for high FK grade level ({max_grade_level_row['fk_grade_level']:.2f}):")
print(max_grade_level_row['final_comment'])
print("-" * 20)


Example near grade level 0 (actual: -0.08):
OP.  Watch them clash.  And they all swear to know the TRUTH.  Must be nice to be so confident.
--------------------
Example near grade level 5 (actual: 4.88):
If it's "parked" in an account then it's fueling the economy through lending. The only way for money to be "parked" is if you take a wad of cash and put it into a safe. No sane man would do that due to inflation.
--------------------
Example near grade level 9 (actual: 9.09):
And the good parts for most people are temporary - the tax breaks for corporations are permanent.
--------------------
Example near grade level 14 (actual: 14.25):
How dare you Venezuela is a utopia and anything that says otherwise is capitalist propaganda.  /s
--------------------
Example near grade level 18 (actual: 17.86):
If you are eliminating the current private sector entirely and installing a single government entity in its stead that is the definition of “nationalization”.
--------------------
Example for

### Flesh Kincaid Reading Ease

In [148]:
comments_df['fk_reading_ease'] = None

In [ ]:
common_functions.do_calc_feature(comments_df, "fk_reading_ease", lambda text: textstat.flesch_reading_ease(text), remove_urls=True)

In [150]:
comments_df['fk_reading_ease'].head()

0      61.3275
1    62.398647
2      116.145
3        63.77
4    57.308092
Name: fk_reading_ease, dtype: object

In [151]:
comments_df['fk_reading_ease'] = pd.to_numeric(comments_df['fk_reading_ease'], errors='coerce')

In [152]:
comments_df['fk_reading_ease'].describe()

count    3971.000000
mean       61.089182
std        18.761663
min      -218.195000
25%        52.307500
50%        61.633891
75%        70.418749
max       121.220000
Name: fk_reading_ease, dtype: float64

Like the grade level minimum above, reading ease also has a long tail of very low (negative) scores. Let's inspect the comments below -20:

In [ ]:
comments_df[comments_df['fk_reading_ease'] < -20]

In [ ]:
common_functions.plot_feature(comments_df, 'fk_reading_ease', 'Flesh Kincaid Reading Ease', 20)

**Insight:** reading ease shows roughly the mirror image of the grade-level pattern (the two metrics are inversely related by construction). Both tails underperform: the hardest-to-read comments (ease below ~20, mostly very short comments packed with rare or long words - see the inspection above) convince at ~7% or less, and the easiest-to-read comments (ease above ~80, i.e. very short and simplistic) convince at only ~6%, dropping to ~1% at the very easiest bucket. The bulk of the data (ease 42-82, over 80% of comments) sits in between, and even there the moderately-harder half (42-62, ~20% convincing) out-converts the easier half (62-82, ~14% convincing). As with grade level, moderate difficulty - neither too simple nor too complex - is associated with the highest persuasion rate.

In [159]:
min_row = comments_df.loc[comments_df['fk_reading_ease'].idxmin()]
print(f"Example for very low FK reading ease ({min_row['fk_reading_ease']:.2f}):")
print(min_row['final_comment'])
print("-" * 20)

max_ease = comments_df['fk_reading_ease'].max()
max_rows = comments_df[comments_df['fk_reading_ease'] == max_ease]
print(f"Example for very high FK reading ease ({max_ease:.2f}, {len(max_rows)} comments tied at this score):")
print(max_rows.iloc[0]['final_comment'])
print("-" * 20)

# Examples from the two most common buckets, for contrast with the extremes above
common_bucket_targets = [55, 70]
for target in common_bucket_targets:
    candidates = comments_df[
        comments_df['fk_reading_ease'].between(target - 3, target + 3) &
        comments_df['final_comment'].str.len().between(80, 260)
    ]
    if len(candidates):
        row = candidates.iloc[0]
        print(f"Example near reading ease {target} (actual: {row['fk_reading_ease']:.2f}):")
        print(row['final_comment'])
        print("-" * 20)


Example for very low FK reading ease (-218.19):
"analyzational"

Analytical?
--------------------
Example for very high FK reading ease (121.22, 5 comments tied at this score):
YES
--------------------
Example near reading ease 55 (actual: 57.95):
There certainly are other arguments against it. But if a plan isn't achievable, I'd say that's problem #1 with the plan.
--------------------
Example near reading ease 70 (actual: 72.27):
The other thing is that your view is really not a useful one. You can always find a technically positive small piece of a large bill and then say he's doing good. But the only real way to evaluate his impact is holistically.
--------------------


In [155]:
comments_df.to_csv('../results/comments_df_readability_checkpoint.csv', index=False)

## 6.7 Use of Evidence

The paper, "[Attitude Change on Reddit's Change My View](https://jpriniski.github.io/papers/cogsci-reddit.pdf)," explores how individuals on the Reddit forum **Change My View (CMV)** adjust their beliefs on various topics, particularly focusing on sociomoral issues (e.g., politics, morality) compared to non-sociomoral topics (e.g., humor, fiction).

The studuy includes the code for calculating evidence use in comments. We will adapt the code from their [repo in GitHub](https://github.com/jpriniski/CMV), and execute it on our own dataset, to create the evidence use features.

`evidence_count` is a simple keyword-stem heuristic, adapted from the paper's own method: it stems every word in the thread + comment text and counts how many match a fixed list of stems associated with citing data or documented facts (`stat`, `percent`, `figur`, `evid`, `document`, `measur`, `\$`, `million`, comparison words like `greater`/`less`, and similar). A **high** value means the comment leans heavily on numbers, statistics, or references to documented/measurable facts to make its case; a **low** value (0 is both the mode and median - about 63% of comments score 0) means the argument is built on reasoning, opinion, or anecdote without citing anything quantifiable. It's a surface-level lexical count, not real fact-checking - a comment can score high by discussing money or percentages loosely, without actually citing a source.

In [ ]:
print("EVIDENCE_STEMS:", common_functions.EVIDENCE_STEMS)
print("EVIDENCE_URL_EXTENSIONS:", common_functions.EVIDENCE_URL_EXTENSIONS)
print()
import inspect
print(inspect.getsource(common_functions.get_evidence_count))

get_evidence_count = common_functions.get_evidence_count

In [161]:
comments_df['evidence_count'] = None

In [ ]:
common_functions.do_calc_feature(comments_df, "evidence_count", get_evidence_count)

In [163]:
comments_df['evidence_count'].describe()

count     3971
unique      21
top          0
freq      2512
Name: evidence_count, dtype: int64

In [164]:
comments_df['evidence_count'] = pd.to_numeric(comments_df['evidence_count'])

In [165]:
comments_df['evidence_count'].describe()

count    3971.000000
mean        0.869302
std         1.911464
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max        31.000000
Name: evidence_count, dtype: float64

In [ ]:
common_functions.plot_feature(comments_df, 'evidence_count', 'Evidence Count', 3)

**Insight:** unlike grade level and reading ease, this feature shows a clean, roughly monotonic relationship with persuasion. Comments with no evidence markers at all (evidence_count=0, ~63% of the data) convince only ~12.5% of the time - well below the ~16% overall average. The rate jumps to ~19% with just one evidence marker, and climbs further to ~23-29% from two markers onward, settling around ~25-28% for anything beyond that. So citing even a little bit of data or documented fact is associated with meaningfully higher persuasiveness than citing none - though it isn't a guarantee: the two comments tied for the highest score in the whole dataset (evidence_count=31) were both non-convincing, so a heavy use of these keywords alone doesn't automatically make an argument work.

In [168]:
max_evidence = comments_df['evidence_count'].max()
max_rows = comments_df[comments_df['evidence_count'] == max_evidence]
print(f"Example for high evidence count ({max_evidence:.0f}, {len(max_rows)} comments tied at this score):")
print(max_rows.iloc[0]['thread_text'])
print(max_rows.iloc[0]['final_comment'])
print("-" * 20)

# Contrast: a typical comment with no evidence markers at all (the mode - about 63% of comments)
no_evidence_row = comments_df[comments_df['evidence_count'] == 0].iloc[10]
print(f"Example for no evidence at all (evidence_count=0):")
print(no_evidence_row['final_comment'])
print("-" * 20)


Example for high evidence count (31, 2 comments tied at this score):
2. I think it's good to use other countries as a barometer for how much we spend. As a percentage of GDP we spend less than Russia and Saudi Arabia. As a total amount of spending we spend slightly more than Saudi Arabia, Russia, and China combined. The amounts the rest of NATO spends is negligible by comparison. We are spending like 10-20% more than would equal out to all of these countries but that's forgetting that we are fighting very real conflicts around the world. We can reduce our military spending but other NATO countries need to start pulling their own weight before we do.

3. The Military Industrial complex is very important. Part of the reason our economy does well is we're pumping money into it via military spending. The money that goes into making bullets and missiles doesn't evaporate when the they're fired, it goes into the pockets of US defense contractors. When our equipment gets old we sell it off to

## 6.8 Use of Persuasive Language

We want to identify if the comment and context include use of persuasive language. To do this, we will use a HuggingFace model: https://huggingface.co/chreh/persuasive_language_detector.

From the model card:
> Given a sentence, our model predicts whether or not the sentence contains "persuasive" language, or language designed to elicit emotions or change readers' opinions. The model was tuned on the SemEval 2020 Task 11 dataset. However, we preprocessed the dataset to adapt it from multilabel technique classification and span-classification to our binary classification task.

The model has two flavours, BERT and XLM-RoBERTa. Based on the documentation, the latter performs better and faster, so we will use it.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
model = AutoModelForSequenceClassification.from_pretrained("chreh/persuasive_language_detector", revision="roberta")


In [ ]:
import torch

max_length = 512
def classify_text(text):
    # The model gets only up to 512 tokens
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=max_length)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predicted_class = torch.argmax(logits, dim=1).item()

    return predicted_class


In [ ]:
text = comments_df.iloc[0]['final_comment']
classify_text(text)

In [ ]:
text

Let's apply the featute on all rows:

In [ ]:
comments_df['use_of_persuasive_lang'] = None

In [ ]:
common_functions.do_calc_feature(comments_df, "use_of_persuasive_lang", classify_text, switch_order=True)

In [ ]:
comments_df['use_of_persuasive_lang']

In [ ]:
common_functions.plot_category_histogram(comments_df, 'use_of_persuasive_lang', ' Use of Persuasive Language')

In [ ]:
# convert comments_df['use_of_persuasive_lang'] to ints
comments_df['use_of_persuasive_lang'] = comments_df['use_of_persuasive_lang'].apply(lambda x: 1 if x == '1' else 0)

## 6.9 Document Embedding

We want that our model will also get some kind of representation of the original text, not only features extracted from it. Therefore we will include the docuemnt embedding in the features list.

First we will calculate the embedding using a variation of the BERT model, bert-base-uncased, which is a well-studied, strong contextual embeddings, suitable for general English NLP tasks.

This model provides encoding of dimension 768.

In [ ]:
import torch
import numpy as np
from transformers import AutoModel, AutoTokenizer

# Load model and tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_embedding(text, chunk_size=256, overlap=128):
    """
    Convert a long document into a single embedding using BERT with chunking.

    Parameters:
    - text (str): The long document
    - chunk_size (int): Max tokens per chunk (default: 256)
    - overlap (int): Overlapping tokens between chunks (default: 128)

    Returns:
    - Aggregated document embedding (numpy array of shape [768])
    """
    # Tokenize document without truncation
    tokens = tokenizer(text, return_tensors="pt", truncation=False)["input_ids"][0]

    # Split into chunks with overlap
    chunk_embeddings = []
    for i in range(0, len(tokens), chunk_size - overlap):
        chunk = tokens[i : i + chunk_size]  # Get chunk

        encoded_input = {
            "input_ids": chunk.unsqueeze(0),  # Add batch dimension
            "attention_mask": torch.ones_like(chunk).unsqueeze(0)  # Mask for valid tokens
        }

        # Forward pass through BERT
        with torch.no_grad():
            outputs = model(**encoded_input)

        # Extract last hidden state
        token_embeddings = outputs.last_hidden_state  # Shape: [1, chunk_size, 768]
        chunk_embedding = token_embeddings.mean(dim=1)  # Mean pooling over tokens

        # Store chunk embedding
        chunk_embeddings.append(chunk_embedding.squeeze().numpy())

    # Aggregate all chunk embeddings (mean pooling over chunks)
    document_embedding = np.mean(chunk_embeddings, axis=0)

    return document_embedding

In [ ]:
# Example Usage
long_text = comments_df.iloc[0]['final_comment']
embedding = get_embedding(long_text)
print(embedding.shape)  # Expected output: (768,)

In [ ]:
comments_df['embedding'] = None

In [ ]:
common_functions.do_calc_feature(comments_df, "embedding", get_embedding)

### Embedding Dimension Reduction with PCA
Since the size of the encoding is very big, and adding so many features to our models will not be practical, we will apply PCA on the embeddings.

To decide the PCA dimension, we will execute some statistical computation to understand what is the minimal dimension that will allow us to keep at least 85% of the information that the original embedding contains.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np

# Extract all embeddings into a matrix
embedding_matrix = np.vstack(comments_df["embedding"].values)  # Shape: (num_samples, 768)

# Apply PCA
pca = PCA().fit(embedding_matrix)  # Fit PCA on the full embedding

# Compute cumulative explained variance
explained_variance = np.cumsum(pca.explained_variance_ratio_)

# Find the number of components that explain at least 85% variance
target_dim = np.argmax(explained_variance >= 0.85) + 1  # +1 because indexing starts at 0

# Plot the explained variance
plt.figure(figsize=(8, 5))
plt.plot(np.arange(1, len(explained_variance) + 1), explained_variance, marker="o")
plt.axhline(y=0.85, color="r", linestyle="--", label="85% variance threshold")
plt.axvline(x=target_dim, color="g", linestyle="--", label=f"Optimal Dim = {target_dim}")
plt.xlabel("Number of PCA Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Explained Variance")
plt.legend()
plt.show()

print(f"Optimal number of components: {target_dim}")


Now we will apply the PCA on the embedding:

In [ ]:
pca = PCA(n_components=target_dim)
reduced_embeddings = pca.fit_transform(embedding_matrix)

# Convert back to DataFrame and merge
embedding_columns = [f"embedding_{i}" for i in range(target_dim)]
embedding_df = pd.DataFrame(reduced_embeddings, columns=embedding_columns)

# Merge with original dataset
# comments_df = comments_df.drop(columns=["embedding"]).reset_index(drop=True)
comments_df = pd.concat([comments_df, embedding_df], axis=1)

## Checkpoint — for `03.3`

Section 6 (all per-comment features) is done. `03.3` picks up from here to compute the same style of features for the comments' original posts (OP), then combines everything into the final `cmv_comments_df.csv`.

In [ ]:
comments_df.to_csv('../results/comments_df_for_03_3_checkpoint.csv', index=False)

import json
with open('../results/untrusted_features_for_03_3_checkpoint.json', 'w') as f:
    json.dump(untrusted_features, f, indent=2)

print(f'Saved comments_df_for_03_3_checkpoint.csv ({len(comments_df)} rows), untrusted_features_for_03_3_checkpoint.json: {untrusted_features}')